# MÍMIR v2 — Google Colab Training Test

Tests the training pipeline on a free **T4 (16GB)** Colab GPU.

**How to use:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run cells top to bottom
3. The crash test will tell you if your batch size fits in VRAM
4. If it passes, kick off real training from the last cell

> ⚠️ Free Colab sessions disconnect after 12h. Use the checkpoint resume logic if you need more time.

## 1. Environment Setup

In [ ]:
# Verify we have a GPU
!nvidia-smi

In [ ]:
# Clone the repo
!git clone https://github.com/pmall/mimir.git
%cd mimir

In [ ]:
# Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']

In [ ]:
# Sync dependencies (this takes 2-5 minutes)
!uv sync

## 2. Hugging Face Auth & ESM-3 Weights

You need a Hugging Face account with ESM-3 access granted at:
https://huggingface.co/EvolutionaryScale/esm3-sm-open-v1

In [ ]:
!hf auth login

In [ ]:
# Download ESM-3 weights (~2.8GB, cached in ~/.cache/huggingface)
!uv run scripts/download_weights.py

## 3. Upload Dataset

Upload your `data/run78-v2/` directory (containing `config.json`, fingerprints LMDB, binders LMDB) using the file browser on the left, or mount Google Drive:

In [ ]:
# Option A: Mount Google Drive (if your data is there)
from google.colab import drive
drive.mount('/content/drive')

# Then symlink the data directory:
# !ln -s /content/drive/MyDrive/mimir_data/run78-v2 data/run78-v2

In [ ]:
# Verify dataset is accessible
import os
CONFIG_PATH = "data/run78-v2/config.json"  # ← update if your path differs
assert os.path.exists(CONFIG_PATH), f"Config not found: {CONFIG_PATH}"
print(f"Config found: {CONFIG_PATH}")

## 4. VRAM Crash Test

Runs **one synthetic epoch** through the actual training loop with max-length sequences.
This tells you the maximum batch size that fits in 16GB VRAM.

Start at `batch_size=4`. If it passes without OOM, try 8, 16, etc.

- `--no-compile` skips `torch.compile` so the crash test runs fast (~1 min instead of ~10 min)

In [ ]:
BATCH_SIZE = 4   # ← start here, increase until you get OOM
ACCUM = 1

!uv run python -m scripts.train_crash_test \
    --batch-size {BATCH_SIZE} \
    --accum {ACCUM}

## 5. Training

Use `--no-compile` on T4 — `torch.compile` takes 10+ minutes to trace on first epoch and
the T4 gains from it are modest. Skip it for Colab validation runs.

Set `--batch-size` to the maximum value that passed the crash test above.
Set `--gradient-accumulation-steps` so that `batch_size × accum ≈ 128`.

Example for batch_size=4: `accum = 128 / 4 = 32`

In [ ]:
BATCH_SIZE = 4        # ← max batch from crash test above
ACCUM = 32            # ← 128 / BATCH_SIZE
CONFIG = "data/run78-v2/config.json"
CHECKPOINT_DIR = "runs/colab_run"
EPOCHS = 5            # ← small for validation; use 500 in production

!uv run python -m scripts.train \
    --config {CONFIG} \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --gradient-accumulation-steps {ACCUM} \
    --lam 0.25 \
    --use-8bit-adam \
    --peak-lr 1e-4 \
    --no-compile \
    -v

## 6. Download Checkpoints

In [ ]:
# Zip and download the run directory
!zip -r colab_run.zip runs/colab_run/

from google.colab import files
files.download('colab_run.zip')